In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import numpy as np
import torch.nn.utils.prune as prune

In [2]:

def get_cifar10_loaders(batch_size=64):
    transform_train = transforms.Compose([
        transforms.Resize(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.4914, 0.4822, 0.4465],
                             [0.2023, 0.1994, 0.2010])
    ])
    transform_test = transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize([0.4914, 0.4822, 0.4465],
                             [0.2023, 0.1994, 0.2010])
    ])

    train_dataset = datasets.CIFAR10(root='./data', train=True,
                                     download=True, transform=transform_train)
    test_dataset  = datasets.CIFAR10(root='./data', train=False,
                                     download=True, transform=transform_test)

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size,
                                               shuffle=True,  num_workers=2)
    test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=batch_size,
                                               shuffle=False, num_workers=2)
    return train_loader, test_loader

In [3]:
def load_googlenet(num_classes=10):
    model = models.googlenet(pretrained=False, aux_logits=True,
                             num_classes=num_classes)
    return model

In [4]:
def train_model(model, train_loader, test_loader, epochs=15, lr=1e-3, device='cpu'):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    print(f"\n{'='*60}")
    print(f"  Training GoogLeNet on CIFAR-10  |  {epochs} epochs")
    print(f"{'='*60}")

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total   = 0
        total_batches = len(train_loader)

        print(f"\n  Epoch [{epoch+1:02d}/{epochs}]")
        print(f"  {'-'*50}")

        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            outputs = model(images)

            if hasattr(outputs, 'logits'):
                loss = criterion(outputs.logits, labels)
                if outputs.aux_logits2 is not None:
                    loss += 0.3 * criterion(outputs.aux_logits2, labels)
                if outputs.aux_logits1 is not None:
                    loss += 0.3 * criterion(outputs.aux_logits1, labels)
                preds = outputs.logits
            else:
                loss  = criterion(outputs, labels)
                preds = outputs

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted  = preds.max(1)
            total        += labels.size(0)
            correct      += predicted.eq(labels).sum().item()

            # ── per-batch update every 50 batches ──
            if (batch_idx + 1) % 50 == 0 or (batch_idx + 1) == total_batches:
                batch_acc  = 100. * correct / total
                avg_loss   = running_loss / (batch_idx + 1)
                pct        = 100. * (batch_idx + 1) / total_batches
                filled     = int(pct // 2)
                bar        = '█' * filled + '░' * (50 - filled)
                print(f"  [{bar}] {pct:5.1f}%  "
                      f"Batch [{batch_idx+1:04d}/{total_batches}]  "
                      f"Loss: {avg_loss:.4f}  "
                      f"Acc: {batch_acc:.2f}%",
                      end='\r', flush=True)

        # ── end-of-epoch summary ──
        train_acc = 100. * correct / total
        val_acc   = evaluate(model, test_loader, device)
        avg_loss  = running_loss / total_batches
        lr_now    = scheduler.get_last_lr()[0]

        print()  # newline after \r
        print(f"  {'─'*50}")
        print(f"  ✔ Train Loss : {avg_loss:.4f}")
        print(f"  ✔ Train Acc  : {train_acc:.2f}%")
        print(f"  ✔ Val Acc    : {val_acc:.2f}%")
        print(f"  ✔ LR         : {lr_now:.6f}")

        scheduler.step()

    print(f"\n{'='*60}")
    print("  Training complete.")
    print(f"{'='*60}\n")
    return model

In [5]:
def evaluate(model, test_loader, device):
    model.eval()
    correct = 0
    total   = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            if hasattr(outputs, 'logits'):
                outputs = outputs.logits
            _, predicted = outputs.max(1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return 100. * correct / total

In [6]:

def get_sparsity_per_layer(model):
    """
    Returns dict: { layer_name: sparsity_ratio (0.0–1.0) }
    and prints a full report.
    """
    layer_sparsity = {}
    total_params   = 0
    zero_params    = 0

    print(f"\n{'='*65}")
    print(f"  Layer-wise Sparsity Report")
    print(f"{'='*65}")
    print(f"  {'Layer':<45} {'Zeros':>7} {'Total':>9} {'Sparsity':>9}")
    print(f"  {'-'*45} {'-'*7} {'-'*9} {'-'*9}")

    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            w = module.weight.data
            t = w.numel()
            z = (w == 0).sum().item()
            total_params += t
            zero_params  += z
            ratio = z / t
            layer_sparsity[name] = ratio
            print(f"  {name:<45} {z:>7} {t:>9} {ratio*100:>8.2f}%")

    overall = 100. * zero_params / total_params if total_params > 0 else 0.
    print(f"\n  {'OVERALL':<45} {zero_params:>7} {total_params:>9} {overall:>8.2f}%")
    print(f"{'='*65}\n")
    return layer_sparsity, overall

In [7]:

def global_unstructured_pruning(model, global_sparsity=0.5):
    """
    Apply L1 global unstructured pruning across all Conv2d + Linear weights.
    Returns dict of per-layer sparsity ratios AFTER pruning.
    """
    # Collect all (module, 'weight') pairs
    parameters_to_prune = []
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            parameters_to_prune.append((module, 'weight'))

    prune.global_unstructured(
        parameters_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=global_sparsity
    )

    # Make pruning permanent (remove re-param hooks, bake mask into weight)
    for module, _ in parameters_to_prune:
        prune.remove(module, 'weight')

    print(f"\n{'='*60}")
    print(f"  Global Unstructured Pruning done  |  target = {global_sparsity*100:.0f}%")
    print(f"{'='*60}")

    layer_sparsity, overall = get_sparsity_per_layer(model)
    return layer_sparsity, overall

In [8]:
def maintain_sparsity(model, layer_sparsity):
    """
    After fine-tuning the model can partially recover zeroed weights.
    Re-apply per-layer masks by re-pruning each layer to the SAME
    sparsity ratio that was measured after global unstructured pruning.
    This keeps the topology identical.
    """
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            if name in layer_sparsity:
                ratio = layer_sparsity[name]
                if ratio > 0.0:
                    prune.l1_unstructured(module, name='weight', amount=ratio)
                    prune.remove(module, 'weight')

In [9]:

def fine_tune(model, train_loader, test_loader,
              layer_sparsity=None, epochs=5, lr=1e-4, device='cpu'):
    """
    Fine-tune the pruned model. After every epoch re-applies sparsity
    masks so pruned weights never drift back.
    """
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    print(f"\n{'='*60}")
    print(f"  Fine-Tuning  |  {epochs} epochs  |  maintaining sparsity")
    print(f"{'='*60}")

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total   = 0

        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            outputs = model(images)
            if hasattr(outputs, 'logits'):
                loss  = criterion(outputs.logits, labels)
                if outputs.aux_logits2 is not None:
                    loss += 0.3 * criterion(outputs.aux_logits2, labels)
                if outputs.aux_logits1 is not None:
                    loss += 0.3 * criterion(outputs.aux_logits1, labels)
                preds = outputs.logits
            else:
                loss  = criterion(outputs, labels)
                preds = outputs

            loss.backward()
            optimizer.step()

            # Re-apply sparsity mask after each step
            if layer_sparsity is not None:
                maintain_sparsity(model, layer_sparsity)

            running_loss += loss.item()
            _, predicted  = preds.max(1)
            total        += labels.size(0)
            correct      += predicted.eq(labels).sum().item()

        train_acc = 100. * correct / total
        val_acc   = evaluate(model, test_loader, device)
        print(f"  Epoch [{epoch+1:02d}/{epochs}]  "
              f"Loss: {running_loss/len(train_loader):.4f}  "
              f"Train Acc: {train_acc:.2f}%  "
              f"Val Acc: {val_acc:.2f}%")

    print(f"{'='*60}\n")
    return model

In [10]:
def get_conv_layers(model):
    conv_layers = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            conv_layers.append((name, module))
    return conv_layers


def compute_importance_matrix(conv_layer):
    """
    G_{c,r,s} = sum_{i=1}^{n} |F_i^{(c,r,s)}|
    Weight shape: (n, C, R, S) → G shape: (C, R, S)
    """
    W = conv_layer.weight.data   # (n, C, R, S)
    G = W.abs().sum(dim=0)       # → (C, R, S)
    return G


def compute_threshold_from_ratio(G, sparsity_ratio):
    """Pick τ so that sparsity_ratio fraction of G values fall below τ."""
    flat = G.flatten().cpu().numpy()
    tau  = float(np.percentile(flat, sparsity_ratio * 100))
    return tau


def generate_mask(G, tau):
    """M_{c,r,s} = 1 if G_{c,r,s} >= τ, else 0"""
    return (G >= tau).float()


def apply_mask_to_layer(conv_layer, M):
    """F_i_pruned = F_i ⊙ M   for all i"""
    with torch.no_grad():
        conv_layer.weight.data *= M.unsqueeze(0)

In [11]:

def cross_filter_structured_pruning(model, layer_sparsity):
    """
    Apply cross-filter structured pruning to every Conv2d.
    Per-layer sparsity ratio = the ratio already measured after
    global unstructured pruning → maintains identical sparsity level.
    """
    conv_layers = get_conv_layers(model)

    print(f"\n{'='*65}")
    print(f"  Cross-Filter Structured Pruning  (per-layer sparsity reused)")
    print(f"{'='*65}")

    for name, layer in conv_layers:
        n, C, R, S = layer.weight.data.shape

        # Use same sparsity ratio recorded for this layer
        sparsity_ratio = layer_sparsity.get(name, 0.5)

        print(f"\n  Layer : {name}")
        print(f"  Shape : n={n}, C={C}, R={R}, S={S}  |  "
              f"sparsity_ratio = {sparsity_ratio*100:.2f}%")

        # Step 2 — importance matrix G
        G   = compute_importance_matrix(layer)

        # Step 3 — threshold τ
        tau = compute_threshold_from_ratio(G, sparsity_ratio)
        print(f"  τ     = {tau:.6f}")

        # Step 4 — binary mask M
        M             = generate_mask(G, tau)
        zeros_in_mask = (M == 0).sum().item()
        total_in_mask = M.numel()
        print(f"  Mask  : {zeros_in_mask}/{total_in_mask} zeros "
              f"({100*zeros_in_mask/total_in_mask:.1f}% of C×R×S positions)")

        # Step 5 — apply: F_pruned = {F1⊙M, F2⊙M, ..., Fn⊙M}
        apply_mask_to_layer(layer, M)

    print(f"\n{'='*65}")
    print("  Cross-filter structured pruning complete.")
    print(f"{'='*65}\n")

In [ ]:
if __name__ == "__main__":

    DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'
    BATCH_SIZE      = 64
    TRAIN_EPOCHS    = 10
    FINETUNE_EPOCHS = 3
    LR_TRAIN        = 1e-3
    LR_FINETUNE     = 1e-4
    SPARSITY_LEVELS = [0.60, 0.70, 0.80, 0.90]

    print(f"\nDevice : {DEVICE}")

    # ── A. Data ───────────────────────────────────────────────────
    train_loader, test_loader = get_cifar10_loaders(BATCH_SIZE)

    # ── B. Load & Train base model ONCE ───────────────────────────
    model = load_googlenet(num_classes=10)
    model.to(DEVICE)
    model = train_model(model, train_loader, test_loader,
                        epochs=TRAIN_EPOCHS, lr=LR_TRAIN, device=DEVICE)

    acc_before_pruning = evaluate(model, test_loader, DEVICE)
    print(f"\n  Accuracy BEFORE pruning : {acc_before_pruning:.2f}%")

    print("\n>>> SPARSITY BEFORE PRUNING:")
    get_sparsity_per_layer(model)

    # ── C. Save trained checkpoint (shared by all sparsity levels) ─
    torch.save(model.state_dict(), "googlenet_trained_base.pth")
    print("  Base checkpoint saved → googlenet_trained_base.pth")

    # ── D. Results collector ───────────────────────────────────────
    all_results = []

    # ═════════════════════════════════════════════════════════════
    # LOOP OVER SPARSITY LEVELS
    # ═════════════════════════════════════════════════════════════
    for GLOBAL_SPARSITY in SPARSITY_LEVELS:

        print(f"\n{'*'*60}")
        print(f"  SPARSITY LEVEL : {GLOBAL_SPARSITY*100:.0f}%")
        print(f"{'*'*60}")

        # ─────────────────────────────────────────────────────────
        # BRANCH 1 — Global Unstructured Pruning → Fine-tune
        # ─────────────────────────────────────────────────────────
        print(f"\n{'#'*60}")
        print(f"  BRANCH 1 : Global Unstructured Pruning  "
              f"| sparsity={GLOBAL_SPARSITY*100:.0f}%")
        print(f"{'#'*60}")

        model_unstructured = load_googlenet(num_classes=10)
        model_unstructured.load_state_dict(
            torch.load("googlenet_trained_base.pth", map_location=DEVICE))
        model_unstructured.to(DEVICE)

        # Apply global unstructured pruning → capture per-layer sparsity
        layer_sparsity, _ = global_unstructured_pruning(
            model_unstructured, global_sparsity=GLOBAL_SPARSITY)

        acc_unstructured_before_ft = evaluate(model_unstructured, test_loader, DEVICE)
        print(f"\n  Acc after unstructured pruning (before fine-tune) : "
              f"{acc_unstructured_before_ft:.2f}%")

        # Fine-tune while maintaining sparsity
        model_unstructured = fine_tune(model_unstructured, train_loader, test_loader,
                                       layer_sparsity=layer_sparsity,
                                       epochs=FINETUNE_EPOCHS, lr=LR_FINETUNE,
                                       device=DEVICE)

        print(f"\n>>> SPARSITY — Unstructured branch (sparsity={GLOBAL_SPARSITY*100:.0f}%):")
        get_sparsity_per_layer(model_unstructured)

        acc_unstructured_final = evaluate(model_unstructured, test_loader, DEVICE)
        print(f"  Acc — Unstructured + Fine-tune (FINAL) : {acc_unstructured_final:.2f}%")

        torch.save(model_unstructured.state_dict(),
                   f"googlenet_unstructured_{int(GLOBAL_SPARSITY*100)}.pth")

        # ─────────────────────────────────────────────────────────
        # BRANCH 2 — Cross-Filter Structured Pruning → Fine-tune
        #            Same per-layer sparsity as Branch 1
        # ─────────────────────────────────────────────────────────
        print(f"\n{'#'*60}")
        print(f"  BRANCH 2 : Cross-Filter Structured Pruning  "
              f"| sparsity={GLOBAL_SPARSITY*100:.0f}%")
        print(f"  (same per-layer sparsity ratios as Branch 1)")
        print(f"{'#'*60}")

        model_crossfilter = load_googlenet(num_classes=10)
        model_crossfilter.load_state_dict(
            torch.load("googlenet_trained_base.pth", map_location=DEVICE))
        model_crossfilter.to(DEVICE)

        # Apply cross-filter pruning using exact same layer_sparsity dict
        cross_filter_structured_pruning(model_crossfilter, layer_sparsity)

        acc_crossfilter_before_ft = evaluate(model_crossfilter, test_loader, DEVICE)
        print(f"\n  Acc after cross-filter pruning (before fine-tune) : "
              f"{acc_crossfilter_before_ft:.2f}%")

        # Fine-tune while maintaining same sparsity
        model_crossfilter = fine_tune(model_crossfilter, train_loader, test_loader,
                                      layer_sparsity=layer_sparsity,
                                      epochs=FINETUNE_EPOCHS, lr=LR_FINETUNE,
                                      device=DEVICE)

        print(f"\n>>> SPARSITY — Cross-filter branch (sparsity={GLOBAL_SPARSITY*100:.0f}%):")
        get_sparsity_per_layer(model_crossfilter)

        acc_crossfilter_final = evaluate(model_crossfilter, test_loader, DEVICE)
        print(f"  Acc — Cross-Filter + Fine-tune (FINAL) : {acc_crossfilter_final:.2f}%")

        torch.save(model_crossfilter.state_dict(),
                   f"googlenet_crossfilter_{int(GLOBAL_SPARSITY*100)}.pth")

        # Collect results for this sparsity level
        all_results.append({
            'sparsity'                  : GLOBAL_SPARSITY,
            'acc_base'                  : acc_before_pruning,
            'acc_unstructured_pruned'   : acc_unstructured_before_ft,
            'acc_unstructured_final'    : acc_unstructured_final,
            'acc_crossfilter_pruned'    : acc_crossfilter_before_ft,
            'acc_crossfilter_final'     : acc_crossfilter_final,
        })

        # Per-level mini summary
        print(f"\n  {'─'*58}")
        print(f"  Quick Summary  |  Sparsity = {GLOBAL_SPARSITY*100:.0f}%")
        print(f"  {'─'*58}")
        print(f"  Unstructured  after prune : {acc_unstructured_before_ft:.2f}%  "
              f"→  after fine-tune : {acc_unstructured_final:.2f}%")
        print(f"  Cross-filter  after prune : {acc_crossfilter_before_ft:.2f}%  "
              f"→  after fine-tune : {acc_crossfilter_final:.2f}%")
        print(f"  Difference (CF - US)      : "
              f"{acc_crossfilter_final - acc_unstructured_final:+.2f}%")
        print(f"  {'─'*58}")

    # ═════════════════════════════════════════════════════════════
    # GRAND COMPARISON TABLE  (all sparsity levels)
    # ═════════════════════════════════════════════════════════════
    print(f"\n\n{'='*78}")
    print("  GRAND COMPARISON TABLE")
    print(f"{'='*78}")
    print(f"  {'Sparsity':>8}  {'Base':>7}  "
          f"{'US-Pruned':>10}  {'US-FT':>7}  "
          f"{'CF-Pruned':>10}  {'CF-FT':>7}  {'CF-US':>7}")
    print(f"  {'-'*8}  {'-'*7}  {'-'*10}  {'-'*7}  {'-'*10}  {'-'*7}  {'-'*7}")

    for r in all_results:
        diff = r['acc_crossfilter_final'] - r['acc_unstructured_final']
        print(f"  {r['sparsity']*100:>7.0f}%  "
              f"{r['acc_base']:>6.2f}%  "
              f"{r['acc_unstructured_pruned']:>9.2f}%  "
              f"{r['acc_unstructured_final']:>6.2f}%  "
              f"{r['acc_crossfilter_pruned']:>9.2f}%  "
              f"{r['acc_crossfilter_final']:>6.2f}%  "
              f"{diff:>+6.2f}%")

    print(f"{'='*78}")
    print("  US-Pruned = Unstructured after pruning (before fine-tune)")
    print("  US-FT     = Unstructured after fine-tune")
    print("  CF-Pruned = Cross-filter after pruning (before fine-tune)")
    print("  CF-FT     = Cross-filter after fine-tune")
    print("  CF-US     = Difference: Cross-filter minus Unstructured (final)")
    print(f"{'='*78}\n")


Device : cuda


100%|██████████| 170M/170M [00:03<00:00, 49.2MB/s]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/torchvision/models/googlenet.py:47: FutureWarning: The default weight initialization of GoogleNet will be changed in future releases of torchvision. If you wish to keep the old behavior (which leads to long initialization times due to scipy/scipy#11299), please set init_weights=True.
  warnings.warn(



  Training GoogLeNet on CIFAR-10  |  10 epochs

  Epoch [01/10]
  --------------------------------------------------

  ──────────────────────────────────────────────────
  ✔ Train Loss : 2.4373
  ✔ Train Acc  : 43.01%
  ✔ Val Acc    : 53.04%
  ✔ LR         : 0.001000

  Epoch [02/10]
  --------------------------------------------------

  ──────────────────────────────────────────────────
  ✔ Train Loss : 1.6666
  ✔ Train Acc  : 64.46%
  ✔ Val Acc    : 66.04%
  ✔ LR         : 0.001000

  Epoch [03/10]
  --------------------------------------------------

  ──────────────────────────────────────────────────
  ✔ Train Loss : 1.3053
  ✔ Train Acc  : 72.99%
  ✔ Val Acc    : 69.20%
  ✔ LR         : 0.001000

  Epoch [04/10]
  --------------------------------------------------

  ──────────────────────────────────────────────────
  ✔ Train Loss : 1.0994
  ✔ Train Acc  : 77.60%
  ✔ Val Acc    : 76.53%
  ✔ LR         : 0.001000

  Epoch [05/10]
  ---------------------------------------------